# Explore métodos de clasificación

## Instrucciones

En la [documentación de Scikit-learn](https://scikit-learn.org/stable/supervised_learning.html) encontrarás un gran lista de formas de clasificar los datos. Haz una pequeña búsqueda en estos documentos: tu meta es buscar los metódos de clasificación y hacer coincidir un conjunto de datos en este plan de estudios, una pregunta que puedes hacerle, y una técnica de clasificación. Crea una hoja de cálculo o tabla en un archivo .doc y explica cómo el conjunto de datos funcionaría con el algoritmo de clasificación.

## Clasificando jugadores de la MLB
Para abordar el modelo de clasificación utilizando bibliotecas como scikit-learn. Supongamos que se desea predecir la "Role" (posición) de un jugador de la MLB en función de las otras características.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE

In [2]:
# Leer el archivo CSV
df = pd.read_csv('Data/SOCR_MLB.tsv',sep='\t', header=None, names=['Name','Team','Role','Height','Weight','Age'])
df

,Name,Team,Role,Height,Weight,Age
0,Adam_Donachie,BAL,Catcher,74,180.0,22.99
1,Paul_Bako,BAL,Catcher,74,215.0,34.69
2,Ramon_Hernandez,BAL,Catcher,72,210.0,30.78
3,Kevin_Millar,BAL,First_Baseman,72,210.0,35.43
4,Chris_Gomez,BAL,First_Baseman,73,188.0,35.71
...,...,...,...,...,...,...
1029,Brad_Thompson,STL,Relief_Pitcher,73,190.0,25.08
1030,Tyler_Johnson,STL,Relief_Pitcher,74,180.0,25.73
1031,Chris_Narveson,STL,Relief_Pitcher,75,205.0,25.19
1032,Randy_Keisler,STL,Relief_Pitcher,75,190.0,31.01


In [3]:
# Verificar la existencia de valores nulos o datos faltantes
print(df.isnull().sum())

Name      0
Team      0
Role      0
Height    0
Weight    1
Age       0
dtype: int64


In [4]:
df['Team'].value_counts()

,count
Team,
NYM,38
OAK,37
ATL,37
DET,37
BOS,36
CIN,36
CHC,36
WAS,36
PHI,36


### Preprocesamiento de datos:

In [5]:
cat_cols = ['Team', 'Role']#df.select_dtypes(include='object').columns

# Creamos un diccionario para guardar los codificadores
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

In [6]:
for col, le in encoders.items():
    print(f"\n🔸 Columna: {col}")
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    for k, v in mapping.items():
        print(f"  {k} → {v}")


🔸 Columna: Team
  ANA → 0
  ARZ → 1
  ATL → 2
  BAL → 3
  BOS → 4
  CHC → 5
  CIN → 6
  CLE → 7
  COL → 8
  CWS → 9
  DET → 10
  FLA → 11
  HOU → 12
  KC → 13
  LA → 14
  MIN → 15
  MLW → 16
  NYM → 17
  NYY → 18
  OAK → 19
  PHI → 20
  PIT → 21
  SD → 22
  SEA → 23
  SF → 24
  STL → 25
  TB → 26
  TEX → 27
  TOR → 28
  WAS → 29

🔸 Columna: Role
  Catcher → 0
  Designated_Hitter → 1
  First_Baseman → 2
  Outfielder → 3
  Relief_Pitcher → 4
  Second_Baseman → 5
  Shortstop → 6
  Starting_Pitcher → 7
  Third_Baseman → 8


In [7]:
# Eliminar filas con valores nulos
df = df.dropna()

In [8]:
df['Role'].value_counts()

,count
Role,
4,315
7,220
3,194
0,76
5,58
2,55
6,52
8,45
1,18


### Entrenar un modelo de clasificación:

In [9]:
# Dividir el conjunto de datos en características (X) y etiquetas (y)
X = df[['Height', 'Weight', 'Age', 'Team']]
y = df['Role']


### Balancear datos con SMOTE

In [10]:
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

print("Distribución de clases después de aplicar SMOTE:")
print(pd.Series(y_res).value_counts())

Distribución de clases después de aplicar SMOTE:
Role
0    315
2    315
5    315
6    315
8    315
3    315
1    315
7    315
4    315
Name: count, dtype: int64


In [11]:
# Dividir el conjunto de datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42)

In [12]:
# Inicializar y entrenar el modelo (Random Forest en este caso)
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

### Hacer predicciones y evaluar el modelo:

In [13]:
# Hacer predicciones en el conjunto de prueba
y_pred = model.predict(X_test)

# Evaluar la precisión del modelo
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')

# Ver informe de clasificación
print(classification_report(y_test, y_pred))

Accuracy: 0.671957671957672
              precision    recall  f1-score   support

           0       0.74      0.75      0.74        60
           1       0.79      0.96      0.86        67
           2       0.68      0.73      0.70        62
           3       0.58      0.44      0.50        66
           4       0.45      0.38      0.41        71
           5       0.72      0.77      0.75        61
           6       0.79      0.76      0.77        63
           7       0.54      0.58      0.56        59
           8       0.70      0.72      0.71        58

    accuracy                           0.67       567
   macro avg       0.67      0.68      0.67       567
weighted avg       0.66      0.67      0.66       567



El uso de este modelo entrenado para hacer predicciones para nuevos datos o hacer preguntas específicas como la prediccion de posicion para nuevos jugadores. 

In [14]:
# Nuevo jugador con estas características
nuevo_jugador = pd.DataFrame({'Height': [72], 'Weight': [80], 'Age': [25], 'Team': [25]})

In [15]:
# Crear un nuevo LabelEncoder solo para la columna 'Team'
le_team = LabelEncoder()

# Ajustar el LabelEncoder al conjunto de entrenamiento original
le_team.fit(df['Team'])

# Codificar el equipo en el nuevo jugador
nuevo_jugador['Team'] = le_team.transform(nuevo_jugador['Team'])

# Hacer la predicción con el modelo entrenado
prediccion = model.predict(nuevo_jugador)

# Convertir el resultado a la etiqueta original utilizando el inverso del LabelEncoder
posicion_predicha = le.inverse_transform(prediccion)

# Mostrar la posición predicha
print(f'La posición predicha para el nuevo jugador es: {posicion_predicha[0]}')

La posición predicha para el nuevo jugador es: Shortstop


In [40]:
roles = df['Role'].unique().tolist()

print("Lista de roles:")
print(roles)

Lista de roles:
['Catcher', 'First_Baseman', 'Second_Baseman', 'Shortstop', 'Third_Baseman', 'Outfielder', 'Designated_Hitter', 'Starting_Pitcher', 'Relief_Pitcher']


Por lo tanto la posición del nuevo jugador es **Shortstop**

Este código dará la posición predicha para el nuevo jugador según las características proporcionadas y el modelo que ha sido entrenado.